In [1]:
#%%
import pandas as pd
from io import StringIO
import json
from pathlib import Path

In [ ]:

def clean_string(x):
    if isinstance(x, str):
        return x.replace("\xa0", "").strip()
    return x

def format(df):
    df = pd.DataFrame(df)
    df = df.T
    df = df[[4,6]]
    df.index = range(0 , len(df))
    df = df.drop([0,1,2])
    df = df.rename(columns={
        4:"Date",
        6:"M2_cn"
    })
    df["Date"] = df["Date"].apply(clean_string)
    # df["Date"] = df["Date"].apply(get_date)
    df["M2_cn"] = df["M2_cn"].apply(clean_string)
    df["Date"] = pd.to_datetime(df["Date"].astype(str))
    df.loc[12,"Date"] = df.loc[12,"Date"].replace(month=10)
    df = df.sort_values(by="Date", ascending=False)
    return df


# bronze_path = Path("/opt/airflow/data/Bronze")
# cn_path = bronze_path / "M2_cn.json"

with open("data/Bronze/M2_cn.json", "r", encoding="utf-8") as f:
    cn = json.load(f)




In [18]:
format(cn[0])

C:\Users\kauan\AppData\Local\Temp\ipykernel_23592\2476426965.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"].astype(str))


,Date,M2_cn
14,2025-12-01,NaN
13,2025-11-01,NaN
12,2025-10-01,3351312.31
11,2025-09-01,3353771.03
10,2025-08-01,3319831.44
9,2025-07-01,3299429.06
8,2025-06-01,3302868.17
7,2025-05-01,3257838.11
6,2025-04-01,3251739.32
5,2025-03-01,3260554.57


In [17]:
cn_format = []

for i in cn:
    cn_format.append(format(i))

M2_cn = pd.concat(cn_format, axis=0, ignore_index=True)
M2_cn.dropna(inplace=True)
M2_cn["M2_cn"] = pd.to_numeric(M2_cn["M2_cn"])*10**9

M2_cn["YearMonth"] = M2_cn["Date"].dt.to_period("M").astype(str)

M2_cn = M2_cn.groupby("YearMonth", as_index=False).agg({
    "M2_cn": "mean"
})
M2_cn = M2_cn.sort_values("YearMonth", ascending=False)

M2_cn

C:\Users\kauan\AppData\Local\Temp\ipykernel_23592\2476426965.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"].astype(str))
C:\Users\kauan\AppData\Local\Temp\ipykernel_23592\2476426965.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"].astype(str))
C:\Users\kauan\AppData\Local\Temp\ipykernel_23592\2476426965.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"].astype(str))
C:\Users\kauan\AppData\Local\Temp\ipykernel_23592\2476426965.py:31: UserWarning: C

,YearMonth,M2_cn
105,2025-10,3.351312e+15
104,2025-09,3.353771e+15
103,2025-08,3.319831e+15
102,2025-07,3.299429e+15
101,2025-06,3.302868e+15
...,...,...
4,2017-05,1.609741e+15
3,2017-04,1.603919e+15
2,2017-03,1.607939e+15
1,2017-02,1.589857e+15
